# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 4: Neural Networks and LLMs

Today we'll work from Traditional ML to Neural Networks to Large Language Models!!

In [1]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR


In [2]:
LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

README.md:   0%|          | 0.00/735 [00:00<?, ?B/s]

c:\Users\anil8\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\anil8\.cache\huggingface\hub\datasets--ed-donner--items_lite. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001.parquet:   0%|          | 0.00/6.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


# Before we look at the Artificial Neural Networks

## There is a different kind of Neural Network we could consider

In [4]:
# Write the test set to a CSV

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [5]:
# Read it back in

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [6]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [7]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")


Human predicted 120.0 for an item that actually costs 219.0


In [8]:
evaluate(human_pricer, test, size=100)

  0%|          | 0/100 [00:00<?, ?it/s]

$99 $184 $12 $15 $18 $10 $119 $135 $6 $270 $643 $329 $15 $26 $24 $18 $29 $25 $25 $53 $35 $126 $25 $127 $273 $398 $55 $6 $101 $51 $30 $5 $35 $9 $10 $419 $25 $11 $186 $33 $161 $51 $23 $155 $150 $4 $31 $18 $115 $82 $25 $111 $410 $75 $67 $34 $8 $10 $122 $28 $116 $17 $19 $60 $599 $60 $160 $355 $75 $34 $17 $2 $70 $76 $41 $9 $226 $5 $5 $4 $0 $7 $5 $74 $7 $10 $68 $74 $5 $3 $17 $45 $5 $16 $0 $153 $2 $122 $150 $355 

# And now - a vanilla Neural Network

During the remainder of this course we will get deeper into how Neural Networks work, and how to train a neural network.

This is just a sneak preview - let's make our own Neural Network, from scratch, using Pytorch.

Use this to get intuition; it's not important to know all about Neural networks at this point..

In [9]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [10]:
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the CountVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [11]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [12]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [13]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [23]:
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# We will do 2 complete runs through the data

EPOCHS = 4

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [1/4], Train Loss: 660.295, Val Loss: 17003.664


  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [2/4], Train Loss: 473.915, Val Loss: 17625.273


  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [3/4], Train Loss: 531.952, Val Loss: 17123.859


  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [4/4], Train Loss: 568.789, Val Loss: 17493.555


In [24]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [25]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$153 $31 $11 $4 $18 $7 $51 $86 $18 $125 $528 $168 $32 $352 $35 $15 $40 $11 $1 $140 $93 $75 $124 $96 $95 $263 $324 $23 $78 $24 $12 $77 $48 $45 $129 $53 $11 $100 $90 $37 $178 $36 $11 $88 $144 $3 $14 $45 $21 $33 $13 $99 $181 $84 $87 $143 $33 $102 $103 $10 $31 $49 $9 $6 $492 $131 $3 $226 $15 $213 $16 $13 $23 $78 $3 $18 $60 $20 $14 $18 $88 $102 $5 $3 $74 $54 $102 $147 $38 $5 $24 $14 $2 $17 $132 $114 $82 $53 $157 $276 $11 $17 $12 $80 $62 $23 $61 $305 $7 $86 $42 $166 $57 $20 $40 $227 $153 $189 $103 $5 $37 $374 $19 $58 $82 $91 $11 $67 $3 $92 $33 $54 $112 $12 $46 $21 $89 $11 $52 $105 $27 $160 $167 $215 $101 $52 $0 $336 $131 $66 $0 $118 $4 $31 $1 $192 $131 $16 $24 $31 $70 $6 $7 $6 $256 $6 $9 $35 $2 $50 $11 $3 $135 $3 $4 $47 $5 $41 $67 $13 $378 $10 $215 $106 $5 $57 $29 $40 $27 $16 $44 $42 $3 $29 $4 $23 $5 $60 $18 $10 

# And now - to the frontier!

Let's see how Frontier models do out of the box; no training, just inference based on their world knowledge.

Tomorrow we will do some training.

In [26]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [27]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [28]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [44]:
# The function for gpt-4.1-nano

def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-5-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [45]:
gpt_4__1_nano(test[0])

'$299'

In [46]:
test[0].price

219.0

In [47]:
evaluate(gpt_4__1_nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$20 $44 $25 $20 $10 $100 $74 $15 $4 $420 $473 $21 $0 $14 $19 $8 $51 $20 $140 $29 $166 $175 $55 $625 $82 $204 $104 $0 $201 $65 $10 $0 $139 $55 $84 $169 $240 $26 $64 $13 $130 $40 $20 $45 $70 $5 $62 $8 $75 $2 $28 $110 $275 $10 $346 $6 $1 $159 $2 $1 $56 $38 $11 $20 $229 $9 $60 $246 $25 $174 $19 $3 $120 $4 $10 $11 $26 $0 $4 $6 $30 $1 $34 $79 $7 $20 $18 $6 $20 $21 $8 $5 $0 $5 $2 $78 $6 $7 $71 $275 $99 $23 $19 $10 $51 $32 $15 $375 $49 $100 $90 $236 $29 $38 $4 $29 $0 $0 $144 $697 $9 $510 $10 $14 $40 $10 $5 $101 $21 $99 $129 $13 $5 $10 $35 $5 $105 $30 $8 $12 $6 $51 $20 $10 $6 $98 $15 $390 $135 $8 $6 $93 $22 $759 $9 $79 $81 $43 $70 $100 $410 $17 $28 $2 $541 $7 $251 $25 $15 $5 $12 $8 $120 $10 $32 $19 $2 $27 $6 $27 $146 $25 $250 $81 $20 $3 $43 $37 $20 $14 $25 $99 $20 $60 $10 $270 $29 $30 $21 $1 

In [37]:
def claude_opus_4_5(item):
    response = completion(model="openrouter/anthropic/claude-opus-4-5", messages=messages_for(item))
    return response.choices[0].message.content

In [39]:
evaluate(claude_opus_4_5, test)

  0%|          | 0/200 [00:00<?, ?it/s]


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



AuthenticationError: litellm.AuthenticationError: AuthenticationError: OpenrouterException - {"error":{"message":"User not found.","code":401}}

In [42]:
def gemini_3_pro_preview(item):
    response = completion(model="gemini/gemini-3-pro-preview", messages=messages_for(item), reasoning_effort='low')
    return response.choices[0].message.content

In [43]:
evaluate(gemini_3_pro_preview, test, size=50, workers=2)

  0%|          | 0/50 [00:00<?, ?it/s]


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



AuthenticationError: litellm.AuthenticationError: geminiException - {
  "error": {
    "code": 400,
    "message": "API key not valid. Please pass a valid API key.",
    "status": "INVALID_ARGUMENT",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.ErrorInfo",
        "reason": "API_KEY_INVALID",
        "domain": "googleapis.com",
        "metadata": {
          "service": "generativelanguage.googleapis.com"
        }
      },
      {
        "@type": "type.googleapis.com/google.rpc.LocalizedMessage",
        "locale": "en-US",
        "message": "API key not valid. Please pass a valid API key."
      }
    ]
  }
}


In [ ]:
def gemini_2__5_flash_lite(item):
    response = completion(model="gemini/gemini-2.5-flash-lite", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
evaluate(gemini_2__5_flash_lite, test)

In [ ]:

def grok_4__1_fast(item):
    response = completion(model="xai/grok-4-1-fast-non-reasoning", messages=messages_for(item), seed=42)
    return response.choices[0].message.content

In [ ]:
evaluate(grok_4__1_fast, test)

In [ ]:
# The function for gpt-5.1

def gpt_5__1(item):
    response = completion(model="gpt-5.1", messages=messages_for(item), reasoning_effort='high', seed=42)
    return response.choices[0].message.content


In [ ]:
evaluate(gpt_5__1, test)

In [54]:
def ollama(item):
    response = completion(model="ollama/llama3.2", base_url="http://localhost:11434", messages=messages_for(item), seed=42)
    return response.choices[0].message.content

In [55]:
evaluate(ollama, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$19 $51 $25 $69 $40 $187 $109 $110 $14 $365 $583 $50 $209 $4 $20 $13 $1 $27 $50 $20 $25 $26 $50 $50 $267 $254 $154 $15 $26 $30 $109 $15 $439 $63 $25 $369 $30 $21 $85 $21 $160 $40 $10 $165 $30 $0 $2 $8 $66 $142 $20 $105 $325 $179 $196 $94 $13 $185 $172 $28 $263 $32 $56 $10 $449 $130 $195 $246 $114 $54 $24 $33 $415 $11 $40 $21 $306 $15 $23 $26 $40 $6 $12 $84 $23 $70 $63 $107 $130 $11 $2 $20 $8 $25 $2 $98 $36 $7 $45 $146 $55 $75 $27 $160 $204 $577 $0 $351 $23 $150 $30 $536 $198 $43 $36 $529 $12 $4 $12 $137 $44 $206 $40 $126 $175 $1 $10 $200 $66 $69 $38 $8 $5 $5 $20 $5 $115 $30 $68 $41 $11 $99 $50 $40 $119 $3 $15 $90 $155 $17 $11 $54 $22 $10 $9 $199 $6 $44 $25 $75 $89 $20 $8 $12 $241 $3 $99 $30 $5 $20 $5 $3 $170 $27 $32 $246 $72 $43 $14 $78 $245 $105 $399 $150 $400 $3 $53 $32 $48 $15 $5 $39 $90 $761 $46 $319 $30 $219 $31 $6 

In [60]:
def ollama_deepseek(item):
    response = completion(model="ollama/deepseek-r1:1.5b", base_url="http://localhost:11434", messages=messages_for(item), seed=42)
    return response.choices[0].message.content

In [61]:
evaluate(ollama_deepseek, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$144 $96 $40 $15 $5 $212 $124 $135 $16 $19670 $1289 $344 $155 $1796 $73 $23 $350 $35 $180 $54 $166 $251 $10 $172 $305 $203 $96 $0 $101 $45 $10 $80 $40 $76 $112 $469 $90 $194 $14 $13 $50 $65 $24965 $180 $170 $21306 $134 $7 $110 $152 $140 $115 $325 $105 $7447 $99 $33 $10 $1813 $23 $116 $222 $85 $108 $479 $2910 $1772 $295 $55 $174 $118 $83 $205 $61 $30 $89 $266 $0 $3 $5981 $10 $3 $570 $74 $20 $1940 $111025 $94 $230 $16 $13 $25 $8 $5 $62 $53 $9 $1154 $20 $15903 $80 $67 $12 $89 $245 $132 $5 $50 $14 $2712 $190 $21 $354 $22 $106 $830 $24 $5 $11 $53 $44 $461 $35 $16 $188 $53 $20 $101 $154 $109 $21 $113 $35 $1991 $20 $50 $105 $230 $48 $111 $47 $235 $40 $80 $164 $268 $30 $1490 $15 $2 $6 $194 $103 $110 $18 $79 $9 $144 $70 $11950 $204 $973 $7 $12 $827 $20 $1352 $30 $39 $10 $40 $178 $145 $2 $48 $124 $122 $4 $19 $407 $154 $20 $1800 $151 $550 $17 $83 $38 $40 $9 $415 $199 $10 $661 $75 $270 $160 $210 $48 $124 